# AraBERT Noun Attachment: Test Sample Inference & Evaluation

## 1. Install Dependencies & Setup Environment

In [ ]:
!pip install -q transformers datasets arabert farasa pyarabic scikit-learn pandas openpyxl torch

In [ ]:
import io
import torch
import numpy as np
import pandas as pd
from google.colab import files
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from arabert.preprocess import ArabertPreprocessor

# Preprocessor base model name and fine-tuned checkpoint path/repo
BASE_MODEL_NAME = "aubmindlab/bert-base-arabertv02"
FINETUNED_MODEL_PATH = "MShormani/arabert-noun-attachment"

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

## 2. Upload Test Sample Dataset

In [ ]:
print("📌 UPLOAD: Select 'test_sample.xlsx'")
up_test = files.upload()
df_test = pd.read_excel(io.BytesIO(up_test['test_sample.xlsx']))

# Map target string labels (N1 -> 0, N2 -> 1)
label_map = {"N1": 0, "N2": 1}
label_col = "human_gold_attachment" if "human_gold_attachment" in df_test.columns else "attachment"
df_test["label"] = df_test[label_col].astype(str).str.upper().str.strip().map(label_map)

print(f"\n✅ Loaded 'test_sample.xlsx' successfully! ({len(df_test):,} rows)\n")

## 3. Preprocessing & Input Sequence Formatting

In [ ]:
print("📌 Preprocessing Arabic text with Farasa & AraBERT Tokenizer...")
arabert_prep = ArabertPreprocessor(model_name=BASE_MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(FINETUNED_MODEL_PATH)

def format_input(row):
    sent = arabert_prep.preprocess(str(row["sentence"]))
    n1 = arabert_prep.preprocess(str(row["head_N1"]))
    n2 = arabert_prep.preprocess(str(row["complement_N2"]))
    return f"{sent} [SEP] N1: {n1} [SEP] N2: {n2}"

df_test["input_text"] = df_test.apply(format_input, axis=1)

class AttachmentDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=128, return_tensors="pt")
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)

test_ds = AttachmentDataset(df_test["input_text"].tolist(), df_test["label"].tolist())

## 4. Load Fine-Tuned Model & Setup Inference

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1_binary, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )
    _, _, f1_weighted, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1_binary": f1_binary,
        "f1_weighted": f1_weighted
    }

print(f"📌 Loading fine-tuned AraBERT model weights from: {FINETUNED_MODEL_PATH}")

model = AutoModelForSequenceClassification.from_pretrained(
    FINETUNED_MODEL_PATH
)

eval_args = TrainingArguments(
    output_dir="./test_sample_results",
    per_device_eval_batch_size=32,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=eval_args,
    compute_metrics=compute_metrics
)

## 5. Model Inference & Test Sample Results

In [ ]:
print("🚀 Running inference on 'test_sample.xlsx'...")
eval_preds = trainer.predict(test_ds)
results = eval_preds.metrics

print("\n==================================================")
print("           TEST SAMPLE EVALUATION METRICS         ")
print("==================================================")
print(f" ACCURACY:     {results.get('test_accuracy', results.get('eval_accuracy', 0)) * 100:.2f}%")
print(f" PRECISION:    {results.get('test_precision', results.get('eval_precision', 0)) * 100:.2f}%")
print(f" RECALL:       {results.get('test_recall', results.get('eval_recall', 0)) * 100:.2f}%")
print(f" BINARY F1:    {results.get('test_f1_binary', results.get('eval_f1_binary', 0)) * 100:.2f}%")
print(f" WEIGHTED F1:  {results.get('test_f1_weighted', results.get('eval_f1_weighted', 0)) * 100:.2f}%")
print("==================================================")

pred_labels = np.argmax(eval_preds.predictions, axis=1)

print("\n--- Detailed Classification Report ---")
print(classification_report(df_test["label"], pred_labels, target_names=["N1", "N2"], digits=4))

cm = confusion_matrix(df_test["label"], pred_labels)
print("--- Confusion Matrix ---")
print(f"True N1 -> Predicted N1: {cm[0][0]:>6} | Predicted N2: {cm[0][1]:>6}")
print(f"True N2 -> Predicted N1: {cm[1][0]:>6} | Predicted N2: {cm[1][1]:>6}")